In [ ]:
import pandas as pd

file_path = r"C:\Users\gabhi\Documents\nsl-kdd\KDDTrain+.txt"

df = pd.read_csv(file_path, header=None)

print("Loaded successfully")
print("Shape:", df.shape)
print(df.head())

: 

In [ ]:
# confirm dataset shape
df.shape

In [ ]:
#preview the data
df.head()

In [ ]:
df.iloc[:, -1].value_counts().head(15)

In [ ]:
#check for duplicate rows
duplicate_count=df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count}")

In [ ]:
print("df.isnull().sum()")

In [ ]:
#add column names
columns = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes",
    "land","wrong_fragment","urgent","hot","num_failed_logins","logged_in",
    "num_compromised","root_shell","su_attempted","num_root","num_file_creations",
    "num_shells","num_access_files","num_outbound_cmds","is_host_login",
    "is_guest_login","count","srv_count","serror_rate","srv_serror_rate",
    "rerror_rate","srv_rerror_rate","same_srv_rate","diff_srv_rate",
    "srv_diff_host_rate","dst_host_count","dst_host_srv_count",
    "dst_host_same_srv_rate","dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate",
    "dst_host_serror_rate","dst_host_srv_serror_rate",
    "dst_host_rerror_rate","dst_host_srv_rerror_rate",
    "label","difficulty"
]

df.columns = columns

df.head()

In [ ]:
#drop unnecessary column
df = df.drop("difficulty", axis=1)

In [ ]:
# convert label to binary: normal (0) vs attack (1)
df["label"] = df["label"].apply(lambda x: 0 if x == "normal" else 1)

df["label"].value_counts()

In [ ]:
# Check for missing values
missing_values = df.isnull().sum()

missing_values[missing_values > 0]

In [ ]:
#check dataset info
df.info()

In [ ]:
#check for duplicates
df.duplicated().sum()

In [ ]:
#drop duplicates
df = df.drop_duplicates()

In [ ]:
df.duplicated().sum()

In [ ]:
# identify categorical columns
categorical_cols = ["protocol_type", "service", "flag"]

df[categorical_cols].head()


In [ ]:
# all categories in protocol_type
df["protocol_type"].value_counts()

In [ ]:
# target variable distribution
df["label"].value_counts()

DATA PREPROCESSING


In [ ]:
# separate categorical and numerical features
categorical_features = [
    "protocol_type",
    "service",
    "flag"
]

numerical_features = [
    col for col in df.columns
    if col not in categorical_features + ["label"]
]

print(f"Categorical: {len(categorical_features)}")
print(f"Numerical: {len(numerical_features)}")

In [ ]:
# split features and target
X = df.drop("label", axis=1)
y = df["label"]

In [ ]:
# split into train and validation sets
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_val.shape)

In [ ]:
# building a preprocessor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

In [ ]:
# data transformation
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)

print(X_train_processed.shape)
print(X_val_processed.shape)

RANDOM FOREST MODEL



In [ ]:
# training random forest classifier
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_processed, y_train)

In [ ]:
# predict on validation set
y_val_pred = rf.predict(X_val_processed)

In [ ]:
# evaluation
from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_val, y_val_pred))

In [ ]:
# classification report
from sklearn.metrics import classification_report

print(classification_report(y_val, y_val_pred))

In [ ]:
# confusion matrix
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_val, y_val_pred))

SAVING MODELS

In [ ]:
# Saving the model and preprocessor
import joblib
from pathlib import Path

# get the path of your current notebook, then move up to the root folder
root_dir = Path.cwd().parent.parent
models_dir = root_dir / "models"

# save the files directly into the root's models directory
joblib.dump(rf, models_dir / "rf_model.pkl")
joblib.dump(preprocessor, models_dir / "preprocessor.pkl")

In [ ]:
# feature importance for report & Linkedln
import pandas as pd

importance = rf.feature_importances_

print(len(importance))

FEATURE ENGINEERING

In [ ]:
# models directory contents
import os

# models directory contents

# ensure models directory exists then list contents
models_dir.mkdir(parents=True, exist_ok=True)
print("Models directory:", models_dir)
print(os.listdir(models_dir))

In [ ]:
# feature importance dataframe
import pandas as pd

feature_names = preprocessor.get_feature_names_out()

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importance
})

importance_df.sort_values(
    by="importance",
    ascending=False
).head(20)

PACKET SNIFFING

In [ ]:
# creating a flow dictionary
from collections import defaultdict
import time

flows = defaultdict(lambda: {
    "packet_count": 0,
    "byte_count": 0,
    "start_time": None,
    "last_seen": None
})

In [ ]:
# updating packet processor
from scapy.layers.inet import IP, TCP, UDP

def process_packet(packet):

    if IP not in packet:
        return

    src_ip = packet[IP].src
    dst_ip = packet[IP].dst
    protocol = packet[IP].proto

    src_port = 0
    dst_port = 0

    if TCP in packet:
        src_port = packet[TCP].sport
        dst_port = packet[TCP].dport

    elif UDP in packet:
        src_port = packet[UDP].sport
        dst_port = packet[UDP].dport

    flow_key = (
        src_ip,
        dst_ip,
        src_port,
        dst_port,
        protocol
    )

    now = time.time()

    flow = flows[flow_key]

    flow["packet_count"] += 1
    flow["byte_count"] += len(packet)

    if flow["start_time"] is None:
        flow["start_time"] = now

    flow["last_seen"] = now

In [ ]:
# running the sniffer
from scapy.all import sniff

sniff(
    prn=process_packet,
    store=False,
    timeout=30
)

In [ ]:
# print flow stats
print("Number of flows:", len(flows))

In [ ]:
for key, value in list(flows.items())[:5]:
    print(key)
    print(value)
    print()

In [ ]:
len(flows)

In [ ]:
# active ethernet interfaces
from scapy.all import IFACES

print(IFACES)

In [ ]:
# verifying if the packet capture is working
from scapy.all import sniff

iface_name = "Ethernet"

selected_iface = None
for iface in IFACES.values():
    if iface.name == iface_name:
        selected_iface = iface.name
        break

if selected_iface is None:
    print(f"Interface '{iface_name}' not found. Available interfaces:")
    for iface in IFACES.values():
        print(f" - {iface.name}")
    raise SystemExit("Please choose a valid interface from the available list.")

print(f"Using interface: {selected_iface}")

sniff(
    iface=selected_iface,
    filter="ip",
    prn=lambda p: print(p.summary()),
    store=False,
    timeout=10
)

In [ ]:
# verifying if the packet capture is working
import pandas as pd

data_file = root_dir / "data" / "processed" / "live_flows.csv"
try:
    df = pd.read_csv(data_file)
except pd.errors.EmptyDataError:
    df = pd.DataFrame()
    print(f"Warning: {data_file} is empty or has no parsable columns.")

print(df.shape)
df.head()

print(df.shape)
df.head()

print(df.shape)
df.head()

In [ ]:
import pandas as pd
data_file = root_dir / "data" / "processed" / "live_flows.csv"
try:
    if not data_file.exists():
        print(f"File not found: {data_file}")
        df = pd.DataFrame()
    elif data_file.stat().st_size == 0:
        print(f"Warning: {data_file} is empty.")
        df = pd.DataFrame()
    else:
        df = pd.read_csv(data_file)
except pd.errors.EmptyDataError:
    df = pd.DataFrame()
    print(f"Warning: {data_file} has no parsable columns (EmptyDataError).")
except Exception as e:
    df = pd.DataFrame()
    print(f"Error reading {data_file}: {e}")

print(df.shape)
print(df.columns.tolist())